# Transformer（2017）

- **核心原理**：Self-Attention、多頭注意力、位置編碼
- **推薦資料集**：Tiny Shakespeare（約 111 萬個字元的莎士比亞劇本；Karpathy 的 char-rnn／nanoGPT 附的資料）
- **實作**：✋ 手刻 —— 注意力的每一個零件先用 NumPy 寫（對照 PyTorch），再用 PyTorch 組成一個 nanoGPT 風格的字元級語言模型，從零訓練
- **論文**：Vaswani et al. (2017), *Attention Is All You Need*；Karpathy, nanoGPT（2022）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第三部分（RNN／LSTM／GRU）的差別：

| | RNN／LSTM／GRU | Transformer |
| --- | --- | --- |
| 怎麼看前文 | 一步一步把資訊塞進隱藏狀態 $h_t$（要經過很多步才傳得到） | **每個位置直接「查詢」所有前面的位置**（一步就到） |
| 計算 | 時間方向必須依序算（不能平行） | 所有位置**同時**算（矩陣乘法，適合 GPU） |
| 梯度 | 沿著時間連乘 → 消失／爆炸（LSTM 用閘門緩解） | 任兩個位置之間只隔一層注意力 |
| 順序資訊 | 天生就有（依序讀） | **沒有** —— 要另外加**位置編碼** |
| 代價 | 每步 $O(1)$ 記憶 | 長度 $T$ 的序列要算 $T \times T$ 的注意力 |

一句話：**Self-attention = 每個位置拿自己的「查詢」（query）去跟所有位置的「鍵」（key）比相似度，再依相似度把那些位置的「值」（value）加權平均 —— 用「內容」決定要看哪裡。**

本章做一個**只有 decoder 的 Transformer**（= GPT 的架構）：輸入前面的字元，預測下一個字元（字元級語言模型）。原論文的 encoder-decoder（翻譯）在第四部分 Seq2Seq＋Attention 與本部分的 T5、BART 會出現。

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3E、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_tiny_shakespeare()` 回傳一個字串。字元級：每個不同的字元一個編號（約 65 個）。切成**連續的**三段：前 80% train、接著 10% validation、最後 10% test（語言模型的資料有順序，跟時間序列一樣不能隨機切）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $V$ | 字元表大小 | 約 65 |
| $B$、$T$ | batch 大小、序列長度（context，`block_size`） | 32、128 |
| $C$ | 模型寬度（embedding 維度，`n_embd`） | 128 |
| $h$、$d_h$ | 注意力頭數、每頭的維度 $C / h$ | 4、32 |
| $L$ | 層數（block 數） | 4 |
| $X$ | 一層的輸入 | $(B, T, C)$ |
| $Q, K, V$ | query、key、value（$XW_Q$、$XW_K$、$XW_V$） | $(B, h, T, d_h)$ |
| $S$ | 注意力分數 $QK^\top/\sqrt{d_h}$ | $(B, h, T, T)$ |
| $A$ | 注意力權重 $\text{softmax}(S + \text{mask})$，每一列加總 1 | $(B, h, T, T)$ |
| PE | 位置編碼 | $(T, C)$ |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第三部分第 1 課 | MLP＋反向傳播 | 線性層 $XW + b$；softmax＋交叉熵；GELU／ReLU；梯度檢查。 |
| 第三部分第 2～4 課 | RNN／LSTM／GRU | `Embedding` 查表；字元級模型（名字分類）；BPTT 的梯度消失；LSTM／GRU 用閘門讓梯度「走捷徑」—— Transformer 的**殘差連接**也是捷徑。 |
| 第一部分第 20 章 | HMM | 「下一個狀態只看上一個」（馬可夫）—— bigram 語言模型就是同樣的假設。 |
| 第一部分第 6 章 | 樸素貝氏 | 數數 + 平滑（$+\alpha$）估計機率；交叉熵、熵。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. softmax**：$\text{softmax}(\mathbf s)_j = e^{s_j}/\sum_k e^{s_k}$。加上 $-\infty$ 的項 → 機率 0（遮罩就是這樣做的）。例：$\mathbf s = (0.707, 0)$ → $(0.670, 0.330)$；$\mathbf s = (0, -\infty)$ → $(1, 0)$。

**2. 內積 = 相似度**：$\mathbf q\cdot\mathbf k$ 大 → 方向接近。$d$ 維、每個元素變異 1 的兩個隨機向量，內積的變異 $= d$ → 除以 $\sqrt d$ 讓變異回到 1（不然 softmax 太尖、梯度很小）。

**3. 語言模型的交叉熵**：$\text{CE} = -\frac1N\sum_t\log P(x_{t+1}\mid x_{\le t})$（nats／字元）。**bits per character** $= \text{CE}/\ln 2$；**perplexity** $= e^{\text{CE}}$ =「平均在幾個字元之間猶豫」。均勻亂猜：$\text{CE} = \ln V \approx 4.17$。

**4. LayerNorm**：對每個位置的 $C$ 維向量減平均、除標準差，再乘 $\gamma$、加 $\beta$（跟 BatchNorm 不同：不看 batch 裡的其他樣本）。

**5. 殘差連接**：$\mathbf x \leftarrow \mathbf x + f(\mathbf x)$。反向傳播時梯度有一條「直接」的路（$\partial\mathbf x/\partial\mathbf x = I$）—— 跟 LSTM 的 cell state 一樣的想法。

**6. 正弦位置編碼**：$\text{PE}[p, 2i] = \sin(p/10000^{2i/C})$、$\text{PE}[p, 2i+1] = \cos(p/10000^{2i/C})$ —— 不同頻率的波，每個位置一個獨特的「指紋」；相對位移是線性變換。

#### 程式

**7. reshape 與 transpose**：$(B, T, C) \to (B, T, h, d_h) \to (B, h, T, d_h)$：`x.reshape(B, T, h, dh).transpose(0, 2, 1, 3)`（NumPy）／`.view(B, T, h, dh).transpose(1, 2)`（PyTorch）。合回來反過來做。
**8. `np.triu(np.ones((T, T)), k=1)`**：嚴格上三角（主對角線以上）= 「未來」的位置。
**9. `@` 對最後兩個維度做矩陣乘法**，前面的維度（$B$、$h$）當成批次。
**10. `torch.multinomial(probs, 1)`**：依機率抽樣。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_tiny_shakespeare

text = load_tiny_shakespeare()
CHARS = sorted(set(text))
V = len(CHARS)
STOI = {c: i for i, c in enumerate(CHARS)}
ITOS = {i: c for c, i in STOI.items()}


def encode(s: str) -> list[int]:
    return [STOI[c] for c in s]


def decode(ids) -> str:
    return "".join(ITOS[int(i)] for i in ids)


data_all = np.array(encode(text), dtype=np.int64)
n = len(data_all)
n_tr, n_va = int(0.8 * n), int(0.1 * n)
data_tr, data_va, data_te = data_all[:n_tr], data_all[n_tr : n_tr + n_va], data_all[n_tr + n_va :]
print(f"{n:,} 個字元；字元表 V = {V}：{''.join(CHARS)!r}")
print(f"train {len(data_tr):,}、validation {len(data_va):,}、test {len(data_te):,}")
print("\n開頭：\n" + text[:300])

In [ ]:
# 字元頻率、行與詞、熵
# 觀察：
#   - 最常見的 10 個字元？空白和換行佔多少？
#   - unigram 的熵是多少 bits？跟「均勻亂猜」的 log2(V) 比？
#   - 一行平均幾個字元？說話者的名字（全大寫 + 冒號）長什麼樣子？
cnt = Counter(text)
freq = pd.Series(cnt).sort_values(ascending=False) / n
print("最常見的 10 個：", {repr(k): round(v, 4) for k, v in freq.head(10).items()})
H_uni = -(freq * np.log2(freq)).sum()
print(f"unigram 熵 {H_uni:.3f} bits／字元；均勻 log2(V) = {np.log2(V):.3f}")
lines = text.split("\n")
print(f"行數 {len(lines):,}；非空行的平均長度 {np.mean([len(l) for l in lines if l]):.1f}；說話者那一行（以 ':' 結尾）的比例 {np.mean([l.endswith(':') for l in lines if l]):.3f}")
words = text.split()
print(f"詞數 {len(words):,}；不同的詞 {len(set(words)):,}；最常見的詞：", Counter(words).most_common(8))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 字元表有幾個字元？大小寫分開嗎？為什麼字元級模型的「詞表」這麼小，卻要學會拼字？（對照：詞級模型的詞表有上萬個）
# (2) unigram 熵約幾 bits？如果只依「字元的頻率」猜下一個字元，交叉熵（nats）是多少？（bits × ln 2）
# (3) 劇本的格式：說話者名字（全大寫 + 冒號）、空行、台詞。一個好的語言模型應該學會哪些「長距離」的結構？（例如：名字之後換行、台詞之後空一行再換說話者）
# (4) 為什麼 train／validation／test 要切成連續的三段，不能隨機抽字元或隨機抽行？
# 答：

---
## 2. Baseline

| Baseline | 下一個字元的機率 | 怎麼估 |
| --- | --- | --- |
| 均勻 | $1/V$ | — |
| unigram | $P(c)$ | 數數 |
| bigram | $P(c_{t+1}\mid c_t)$ | 數數 + add-$\alpha$ 平滑（$\alpha$ 用 validation 選） |
| trigram | $P(c_{t+1}\mid c_{t-1}, c_t)$ | 同上 |
| LSTM（第三部分第 3 課） | 用 PyTorch `nn.LSTM`，第 3 節訓練 | 跟 Transformer 同樣的訓練預算 |

**指標**：validation 的交叉熵（nats／字元），另外換算 bits per character 與 perplexity（前置複習 3）。

In [ ]:
def ngram_ce(train: np.ndarray, eval_: np.ndarray, order: int, alpha: float) -> float:
    """order = 1（unigram）、2（bigram）、3（trigram）的 add-alpha 模型，在 eval_ 上的平均交叉熵（nats）。"""
    shape = (V,) * order
    counts = np.zeros(shape)
    idx = tuple(train[k : len(train) - order + 1 + k] for k in range(order))
    np.add.at(counts, idx, 1)
    probs = (counts + alpha) / (counts + alpha).sum(axis=-1, keepdims=True)
    eidx = tuple(eval_[k : len(eval_) - order + 1 + k] for k in range(order))
    return float(-np.log(probs[eidx]).mean())


val_table = pd.DataFrame(columns=["val CE (nats)", "bits/char", "perplexity"])


def add_row(name: str, ce: float) -> None:
    val_table.loc[name] = [ce, ce / math.log(2), math.exp(ce)]


add_row("uniform", math.log(V))
for order, name in [(1, "unigram"), (2, "bigram"), (3, "trigram")]:
    best = min(((ngram_ce(data_tr, data_va, order, a), a) for a in [0.001, 0.01, 0.1, 1.0]))
    add_row(f"{name} (alpha={best[1]})", best[0])
display(val_table.round(4))

In [ ]:
# TODO 2（問答）
#
# (1) bigram、trigram 比 unigram 好多少？為什麼 n 越大進步越多、但不能一直加大？（提示：V^n 個參數，大部分組合沒出現過 → 需要平滑）
# (2) 最好的 α 在 bigram 和 trigram 一樣嗎？為什麼 trigram 需要比較大（或比較小）的 α？
# (3) 先寫下預測：一個 4 層、約 80 萬參數的 Transformer（context 128 個字元）能到多少 nats？比 trigram 好多少？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | NumPy：scaled dot-product attention | 3 |
| 3B | NumPy：因果遮罩（causal mask） | 4 |
| 3C | NumPy：多頭的切分與合併；對照 PyTorch `nn.MultiheadAttention` | 5、7 |
| 3D | NumPy：正弦位置編碼 | 6 |
| 3E | PyTorch：`CausalSelfAttention`、`Block`、`CharTransformer`、生成；訓練 | 8、9、10、11 |
| 3F | 實驗：拿掉遮罩、拿掉位置編碼、頭數、$1/\sqrt{d}$、注意力圖 | 12 |

### 3A. Scaled dot-product attention

$$\text{Attention}(Q, K, V) = \text{softmax}\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V$$

讀法：第 $i$ 列（位置 $i$）的輸出 = 所有位置的 value 的加權平均，權重 = 「$i$ 的 query 跟各位置的 key 的相似度」的 softmax。

#### 符號表（3A–3D）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $Q, K, V$ | query、key、value（最後兩維是 $T$ 與 $d$；前面可以有批次維度） | $(\dots, T, d)$ | `Q`、`K`、`Vv` |
| $S$ | $QK^\top/\sqrt{d}$ | $(\dots, T, T)$ | `S` |
| mask | 要擋掉的位置是 $-\infty$、其他 0 | $(T, T)$ | `mask` |
| $A$ | softmax 後的權重 | $(\dots, T, T)$ | `A` |

**下標公式**：
- TODO 3：$S_{ij} = \frac1{\sqrt d}\sum_l Q_{il}K_{jl}$ → `Q @ K.swapaxes(-1, -2) / sqrt(d)`；softmax 沿著最後一軸（$j$）：先減掉每列的最大值再 exp（數值穩定）；輸出 `A @ Vv`。

#### 最小例子：TODO 3、4

$Q = K = \begin{pmatrix}1 & 0\\0 & 1\end{pmatrix}$、$V = \begin{pmatrix}1 & 2\\3 & 4\end{pmatrix}$、$d = 2$。

1. **正確答案**（迴圈）：$S = \begin{pmatrix}0.7071 & 0\\0 & 0.7071\end{pmatrix}$；第 0 列 softmax $= (0.6698, 0.3302)$ → 輸出 $0.6698\cdot(1, 2) + 0.3302\cdot(3, 4) = (1.6605, 2.6605)$；第 1 列 $(0.3302, 0.6698)$ → $(2.3395, 3.3395)$。
   **加上因果遮罩**（TODO 4）：位置 0 只能看自己 → 權重 $(1, 0)$、輸出 $(1, 2)$；位置 1 不變。
2. **積木**：`K.swapaxes(-1, -2)`、減掉最大值的 softmax。
3. **留給你**：TODO 3、4。

In [ ]:
# 最小例子：TODO 3、4 —— 1. 正確答案：迴圈
Q_mini = np.eye(2)
K_mini = np.eye(2)
V_mini = np.array([[1.0, 2.0], [3.0, 4.0]])
for causal in [False, True]:
    out_rows = []
    for i in range(2):
        s = np.array([Q_mini[i] @ K_mini[j] / np.sqrt(2) if (not causal or j <= i) else -np.inf for j in range(2)])
        w = np.exp(s - s.max())
        w /= w.sum()
        out_rows.append(w @ V_mini)
    print(f"causal = {causal}：", np.round(out_rows, 4).tolist())
# 寫完 TODO 3、4、執行過下一格之後，取消下面的註解來對照：
# print(attention_np(Q_mini, K_mini, V_mini)[0].round(4), "\n", attention_np(Q_mini, K_mini, V_mini, causal_mask(2))[0].round(4))

In [ ]:
def softmax_np(S: np.ndarray, axis: int = -1) -> np.ndarray:
    S = S - S.max(axis=axis, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=axis, keepdims=True)


def attention_np(Q: np.ndarray, K: np.ndarray, Vv: np.ndarray, mask: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """scaled dot-product attention。回傳 (輸出 (..., T, d_v), 權重 A (..., T, T))。mask：加到分數上（−inf = 擋掉）。"""
    d = Q.shape[-1]
    # TODO 3（程式）：S、（加上 mask）、A = softmax、輸出 A @ Vv
    raise NotImplementedError("TODO 3：attention")


def causal_mask(T: int) -> np.ndarray:
    """(T, T)：j > i（未來）的位置是 −inf，其他 0。"""
    # TODO 4（程式）：用 np.triu(np.ones((T, T)), k=1) 找出「未來」，設成 −inf
    raise NotImplementedError("TODO 4：因果遮罩")

### 3C. 多頭注意力

一組 $Q, K, V$ 只能學一種「看哪裡」的模式。**多頭**：把 $C$ 維切成 $h$ 份，每份 $d_h = C/h$ 維各自做注意力，再接回 $C$ 維，最後乘一個輸出矩陣 $W_O$：

$$\text{MHA}(X) = \text{concat}(\text{head}_1, \dots, \text{head}_h)\,W_O,\qquad \text{head}_i = \text{Attention}(XW_Q^{(i)}, XW_K^{(i)}, XW_V^{(i)})$$

實作上 $W_Q$ 是一個 $(C, C)$ 的矩陣（$h$ 個頭的 $W_Q^{(i)}$ 並排），算完 $XW_Q$ 再 reshape 成 $(h, T, d_h)$（前置複習 7）。

**下標公式**：
- TODO 5：`split_heads`：$(T, C) \to (T, h, d_h) \to (h, T, d_h)$：`X.reshape(T, h, dh).transpose(1, 0, 2)`；`merge_heads` 反過來：`.transpose(1, 0, 2).reshape(T, C)`。

#### 最小例子：TODO 5

$X = \begin{pmatrix}0&1&2&3\\4&5&6&7\end{pmatrix}$（$T = 2$、$C = 4$）、$h = 2$ → 頭 0 = $\begin{pmatrix}0&1\\4&5\end{pmatrix}$、頭 1 = $\begin{pmatrix}2&3\\6&7\end{pmatrix}$（每個頭拿「連續的 $d_h$ 欄」）；合回來 = 原本的 $X$。

In [ ]:
def split_heads(X: np.ndarray, h: int) -> np.ndarray:
    """(T, C) → (h, T, C/h)。"""
    T, C = X.shape
    # TODO 5（程式）之一
    raise NotImplementedError("TODO 5：split_heads")


def merge_heads(Hh: np.ndarray) -> np.ndarray:
    """(h, T, d_h) → (T, h·d_h)。"""
    h, T, dh = Hh.shape
    # TODO 5（程式）之二
    raise NotImplementedError("TODO 5：merge_heads")


def mha_np(X: np.ndarray, Wq, Wk, Wv, Wo, bq, bk, bv, bo, h: int, mask: np.ndarray | None = None) -> np.ndarray:
    """多頭注意力（一個序列）。X (T, C)；W* (C, C)；b* (C,)。"""
    Qh, Kh, Vh = (split_heads(X @ W + b, h) for W, b in [(Wq, bq), (Wk, bk), (Wv, bv)])
    out, _ = attention_np(Qh, Kh, Vh, mask)
    return merge_heads(out) @ Wo + bo


# 寫完 TODO 5 之後，取消下面的註解來對照：
# X_mh = np.arange(8.0).reshape(2, 4); print(split_heads(X_mh, 2), "\n", merge_heads(split_heads(X_mh, 2)))

### 3D. 位置編碼

注意力本身**不知道順序**：把輸入的位置打亂，每個位置的輸出只會跟著打亂（排列等變，permutation equivariant —— 下一格驗證）。所以要把「位置」的資訊加進輸入：

- **正弦位置編碼**（原論文，前置複習 6）：固定的、不用學；
- **可學的位置 embedding**（GPT、nanoGPT）：`nn.Embedding(block_size, C)`，本章的模型用這個。

**下標公式**：
- TODO 6：`pos = np.arange(T)[:, None]`、`i2 = np.arange(0, C, 2)[None, :]`、`angle = pos / 10000 ** (i2 / C)`；`PE[:, 0::2] = sin(angle)`、`PE[:, 1::2] = cos(angle)`。

#### 最小例子：TODO 6

$C = 4$、位置 1：$(\sin 1, \cos 1, \sin 0.01, \cos 0.01) = (0.8415, 0.5403, 0.0100, 0.99995)$；位置 0：$(0, 1, 0, 1)$。

In [ ]:
def sinusoidal_pe(T: int, C: int) -> np.ndarray:
    """(T, C) 的正弦位置編碼。"""
    PE = np.zeros((T, C))
    # TODO 6（程式）：見下標公式
    raise NotImplementedError("TODO 6：位置編碼")
    return PE


# 寫完 TODO 6 之後，取消下面的註解來對照：
# print(sinusoidal_pe(2, 4).round(5))

In [ ]:
# 對照 PyTorch：attention、多頭注意力；驗證「沒有位置編碼 → 排列等變」
import torch
import torch.nn.functional as F
from torch import nn

rng = np.random.default_rng(0)
B_, T_, C_, h_ = 3, 6, 8, 2
Qr, Kr, Vr = (rng.normal(size=(B_, h_, T_, C_ // h_)) for _ in range(3))
for causal in [False, True]:
    mine, _ = attention_np(Qr, Kr, Vr, causal_mask(T_) if causal else None)
    ref = F.scaled_dot_product_attention(*(torch.tensor(a) for a in (Qr, Kr, Vr)), is_causal=causal).numpy()
    print(f"✅ attention（causal = {causal}）vs F.scaled_dot_product_attention：{np.abs(mine - ref).max():.1e}")

mha_t = nn.MultiheadAttention(C_, h_, batch_first=True, dtype=torch.float64)
Wq, Wk, Wv = (w.detach().numpy().T for w in mha_t.in_proj_weight.chunk(3))
bq, bk, bv = mha_t.in_proj_bias.detach().numpy().reshape(3, C_)
Wo, bo = mha_t.out_proj.weight.detach().numpy().T, mha_t.out_proj.bias.detach().numpy()
X_ = rng.normal(size=(T_, C_))
mine = mha_np(X_, Wq, Wk, Wv, Wo, bq, bk, bv, bo, h_, causal_mask(T_))
Xt = torch.tensor(X_)[None]
ref, _ = mha_t(Xt, Xt, Xt, attn_mask=torch.tensor(causal_mask(T_)), need_weights=False)
print(f"✅ 多頭注意力 vs nn.MultiheadAttention（同樣的權重、因果遮罩）：{np.abs(mine - ref[0].detach().numpy()).max():.1e}")

perm = rng.permutation(T_)
out = mha_np(X_, Wq, Wk, Wv, Wo, bq, bk, bv, bo, h_)
out_perm = mha_np(X_[perm], Wq, Wk, Wv, Wo, bq, bk, bv, bo, h_)
print("✅ 沒有遮罩、沒有位置編碼：打亂輸入 = 打亂輸出？", np.allclose(out[perm], out_perm))
PE_ = sinusoidal_pe(T_, C_)
print("   加上位置編碼之後還是嗎？", np.allclose(mha_np(X_ + PE_, Wq, Wk, Wv, Wo, bq, bk, bv, bo, h_)[perm], mha_np(X_[perm] + PE_, Wq, Wk, Wv, Wo, bq, bk, bv, bo, h_)))

In [ ]:
# TODO 7（問答）TODO 3–6 與對照
#
# (1) 手寫的 attention、多頭注意力跟 PyTorch 差多少？nn.MultiheadAttention 的 in_proj_weight 為什麼要 chunk(3) 再轉置？
# (2) 排列等變：沒有位置編碼時，「dog bites man」與「man bites dog」的每個字元得到的表示會怎樣？加上位置編碼之後呢？
# (3) 因果遮罩為什麼對「預測下一個字元」是必要的？如果沒有遮罩，位置 i 的輸出可以看到什麼？（第 3F 節實驗 1 會實際訓練一次）
# (4) 一層注意力的計算量 ∝ T² · C。T 從 128 變 1,024，注意力的計算量變幾倍？RNN 呢？
# 答：

### 3E. PyTorch：組成 Transformer 語言模型

**程式結構圖**：

```
CharTransformer
├── tok_emb : nn.Embedding(V, C)           字元 → 向量
├── pos_emb : nn.Embedding(block_size, C)  位置 → 向量（可學的）
├── blocks  : L 個 Block
│     ├── ln1  : LayerNorm(C)
│     ├── attn : CausalSelfAttention        qkv: Linear(C, 3C)、proj: Linear(C, C)
│     ├── ln2  : LayerNorm(C)
│     └── mlp  : Linear(C, 4C) → GELU → Linear(4C, C)
├── ln_f    : LayerNorm(C)
└── head    : Linear(C, V)                  向量 → 下一個字元的 logits
```

**Block（pre-LN，GPT-2 的做法）**：$\mathbf x \leftarrow \mathbf x + \text{Attn}(\text{LN}_1(\mathbf x))$；$\mathbf x \leftarrow \mathbf x + \text{MLP}(\text{LN}_2(\mathbf x))$。（原論文是 post-LN：先加再 LN；pre-LN 訓練比較穩。）

#### 符號表（3E）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| `idx` | 字元編號 | $(B, T)$ | |
| `x` | 每個位置的向量 | $(B, T, C)$ | |
| `q, k, v` | 切成多頭之後 | $(B, h, T, d_h)$ | |
| `att` | 注意力權重 | $(B, h, T, T)$ | `self.last_att`（存下來畫圖用） |
| `mask` | 下三角為 True 的 $(T_{\max}, T_{\max})$ buffer | | `self.tril` |

**下標公式**：
- TODO 8：`q, k, v = self.qkv(x).split(C, dim=2)`；各自 `.view(B, T, h, dh).transpose(1, 2)`；`att = (q @ k.transpose(-2, -1)) / sqrt(dh)`；`att = att.masked_fill(~self.tril[:T, :T], −inf)`（`use_mask` 為 False 時跳過）；`att = softmax(att, dim=−1)`；`y = (att @ v).transpose(1, 2).reshape(B, T, C)`；`return self.proj(y)`。
- TODO 9：`x = x + self.attn(self.ln1(x))`；`x = x + self.mlp(self.ln2(x))`。
- TODO 10（生成）：迴圈 `n_new` 次：`idx_cond = idx[:, −block_size:]`；`logits = model(idx_cond)[:, −1, :] / temperature`；`probs = softmax(logits)`；`next = argmax（greedy）或 torch.multinomial(probs, 1)`；`idx = torch.cat([idx, next], dim=1)`。

#### 最小例子：TODO 10

一個假的「模型」：下一個字元永遠是「最後一個字元 + 1（mod 5）」（logits 是 one-hot × 10）。從 `[0]` greedy 生成 4 個 → `[0, 1, 2, 3, 4]`。

In [ ]:
from learning_map.utils import count_parameters, get_device, set_seed

device = get_device()  # Mac 會用 MPS（比 CPU 快很多）；沒有 GPU 時用 CPU


class CausalSelfAttention(nn.Module):
    def __init__(self, C: int, n_head: int, block_size: int, use_mask: bool = True, scale: bool = True) -> None:
        super().__init__()
        self.qkv = nn.Linear(C, 3 * C)
        self.proj = nn.Linear(C, C)
        self.n_head, self.use_mask, self.scale = n_head, use_mask, scale
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        h, dh = self.n_head, C // self.n_head
        # TODO 8（程式）：見下標公式（scale 為 False 時不除以 √dh；把 softmax 後的權重存進 self.last_att）
        raise NotImplementedError("TODO 8：CausalSelfAttention")


class Block(nn.Module):
    def __init__(self, C: int, n_head: int, block_size: int, **attn_kw) -> None:
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(C), nn.LayerNorm(C)
        self.attn = CausalSelfAttention(C, n_head, block_size, **attn_kw)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 9（程式）：pre-LN 的兩個殘差（兩行）
        raise NotImplementedError("TODO 9：Block")


class CharTransformer(nn.Module):
    def __init__(self, V: int, C: int = 128, n_layer: int = 4, n_head: int = 4, block_size: int = 128, use_pos: bool = True, **attn_kw) -> None:
        super().__init__()
        self.block_size, self.use_pos = block_size, use_pos
        self.tok_emb = nn.Embedding(V, C)
        self.pos_emb = nn.Embedding(block_size, C)
        self.blocks = nn.ModuleList([Block(C, n_head, block_size, **attn_kw) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(C)
        self.head = nn.Linear(C, V)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx (B, T) → logits (B, T, V)。"""
        T = idx.shape[1]
        x = self.tok_emb(idx)
        if self.use_pos:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))


@torch.no_grad()
def generate(model: nn.Module, idx: torch.Tensor, n_new: int, temperature: float = 1.0, greedy: bool = False, block_size: int = 128) -> torch.Tensor:
    """自回歸生成：每次把最後 block_size 個字元丟進模型，取最後一個位置的分布抽下一個字元。"""
    model.eval()
    for _ in range(n_new):
        # TODO 10（程式）：見下標公式
        raise NotImplementedError("TODO 10：生成")
    return idx


class _PlusOne(nn.Module):
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        return F.one_hot((idx + 1) % 5, 5).float() * 10


# 寫完 TODO 10 之後，取消下面的註解來對照（應為 [[0, 1, 2, 3, 4]]）：
# print(generate(_PlusOne(), torch.tensor([[0]]), 4, greedy=True).tolist())

In [ ]:
# 檢查：手寫的 Block（dropout 0）= PyTorch 的 nn.TransformerEncoderLayer（norm_first=True、GELU、batch_first、因果遮罩）
torch.manual_seed(0)
blk = Block(16, 4, 32).double()
ref = nn.TransformerEncoderLayer(16, 4, dim_feedforward=64, dropout=0.0, activation="gelu", batch_first=True, norm_first=True).double()
with torch.no_grad():
    ref.self_attn.in_proj_weight.copy_(blk.attn.qkv.weight)
    ref.self_attn.in_proj_bias.copy_(blk.attn.qkv.bias)
    ref.self_attn.out_proj.weight.copy_(blk.attn.proj.weight)
    ref.self_attn.out_proj.bias.copy_(blk.attn.proj.bias)
    ref.linear1.weight.copy_(blk.mlp[0].weight), ref.linear1.bias.copy_(blk.mlp[0].bias)
    ref.linear2.weight.copy_(blk.mlp[2].weight), ref.linear2.bias.copy_(blk.mlp[2].bias)
    ref.norm1.load_state_dict(blk.ln1.state_dict()), ref.norm2.load_state_dict(blk.ln2.state_dict())
x_chk = torch.randn(2, 10, 16, dtype=torch.float64)
mask_chk = nn.Transformer.generate_square_subsequent_mask(10, dtype=torch.float64)
print(f"✅ Block vs nn.TransformerEncoderLayer：{(blk(x_chk) - ref(x_chk, src_mask=mask_chk, is_causal=True)).abs().max().item():.1e}")

#### 訓練

- 每一步隨機抽 $B = 32$ 段長度 $T = 128$ 的連續字元：輸入 `x = data[i : i+T]`、目標 `y = data[i+1 : i+T+1]`（**每個位置都在預測下一個字元** → 一段 128 個字元就有 128 個訓練樣本）。
- AdamW、學習率 1e-3（前 100 步 warmup、之後 cosine 遞減）、梯度裁剪 1.0。
- 每 250 步在 validation 上估一次 loss（固定的 40 段，所以不同模型可以直接比）。

（這台電腦的 CPU 上，這個約 80 萬參數的模型每一步約 0.15 秒；Mac 的 MPS 會快很多。）

In [ ]:
def get_batch(data: np.ndarray, B: int, T: int, rng: np.random.Generator) -> tuple[torch.Tensor, torch.Tensor]:
    ix = rng.integers(0, len(data) - T - 1, size=B)
    x = np.stack([data[i : i + T] for i in ix])
    y = np.stack([data[i + 1 : i + T + 1] for i in ix])
    return torch.as_tensor(x, device=device), torch.as_tensor(y, device=device)


VAL_BATCHES = [get_batch(data_va, 32, 128, np.random.default_rng(123)) for _ in range(40)]  # 固定的評估集


@torch.no_grad()
def estimate_loss(model: nn.Module, batches: list, T: int | None = None) -> float:
    model.eval()
    losses = []
    for x, y in batches:
        if T is not None:
            x, y = x[:, :T], y[:, :T]
        losses.append(F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item())
    model.train()
    return float(np.mean(losses))


def train_lm(model: nn.Module, iters: int = 1500, lr: float = 1e-3, B: int = 32, T: int = 128, eval_every: int = 250, seed: int = 0, verbose: bool = True) -> dict:
    set_seed(seed)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s, iters) / iters)))
    rng = np.random.default_rng(seed)
    hist = {"iter": [], "train": [], "val": [], "seconds": 0.0}
    t0, run = time.perf_counter(), []
    for it in range(1, iters + 1):
        x, y = get_batch(data_tr, B, T, rng)
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        run.append(loss.item())
        if it % eval_every == 0 or it == iters:
            hist["iter"].append(it)
            hist["train"].append(float(np.mean(run[-eval_every:])))
            hist["val"].append(estimate_loss(model, VAL_BATCHES, T))
            if verbose:
                print(f"iter {it:5d}  train {hist['train'][-1]:.4f}  val {hist['val'][-1]:.4f}  ({time.perf_counter() - t0:.0f}s)")
    hist["seconds"] = time.perf_counter() - t0
    return hist


set_seed(0)
model = CharTransformer(V)
print("參數量：", count_parameters(model))
hist_main = train_lm(model, iters=1500)
add_row("Transformer 4L/128d (1500 iters)", hist_main["val"][-1])
print("\n生成（temperature 0.8）：\n" + decode(generate(model, torch.as_tensor([encode("ROMEO:\n")], device=device), 300, temperature=0.8)[0].tolist()))

In [ ]:
# 對照：同樣的訓練預算，LSTM（第三部分）
class CharLSTM(nn.Module):
    def __init__(self, V: int, C: int = 256, n_layer: int = 2) -> None:
        super().__init__()
        self.emb = nn.Embedding(V, C)
        self.lstm = nn.LSTM(C, C, n_layer, batch_first=True)
        self.head = nn.Linear(C, V)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        return self.head(self.lstm(self.emb(idx))[0])


set_seed(0)
lstm = CharLSTM(V)
print("LSTM 參數量：", count_parameters(lstm))
hist_lstm = train_lm(lstm, iters=1500, lr=2e-3, verbose=False)
add_row("LSTM 2L/256 (1500 iters)", hist_lstm["val"][-1])
print(f"LSTM：{hist_lstm['seconds']:.0f} 秒；Transformer：{hist_main['seconds']:.0f} 秒")
display(val_table.round(4))
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, h in [("Transformer", hist_main), ("LSTM", hist_lstm)]:
    ax.plot(h["iter"], h["val"], "o-", label=f"{name} val")
    ax.plot(h["iter"], h["train"], "--", label=f"{name} train")
ax.set_xlabel("iteration")
ax.set_ylabel("cross-entropy (nats/char)")
ax.legend()
plt.show()

In [ ]:
# TODO 11（問答）訓練結果
#
# (1) Block vs nn.TransformerEncoderLayer 的差異是多少？
# (2) Transformer 的 validation loss 是多少 nats？換成 bits/char、perplexity？比 trigram 好多少？比 LSTM 呢？（同樣的步數；參數量也比一比）
# (3) train 和 val 的 loss 差多少？有過擬合嗎？
# (4) 讀生成的文字：學會了哪些東西（說話者的格式、換行、拼字、標點）？還沒學會什麼（文法、意義、劇情）？
# 答：

### 3F. 實驗（每個 600 步，跟 3E 的模型同樣大小）

| 實驗 | 問題 |
| --- | --- |
| 1 | **拿掉因果遮罩**：訓練／驗證 loss 會變成多少？生成的文字呢？ |
| 2 | **拿掉位置 embedding** |
| 3 | 頭數 1、4、16（$C = 128$ 不變 → $d_h$ = 128、32、8） |
| 4 | **不除以 $\sqrt{d_h}$**：初始化時 softmax 有多尖？ |
| 5 | 注意力圖：訓練好的模型，各層各頭在看哪裡？ |

In [ ]:
exp_rows = {}
exp_models = {}
for name, kw in {
    "baseline (600 iters)": dict(),
    "no causal mask": dict(use_mask=False),
    "no positional embedding": dict(use_pos=False),
    "1 head": dict(n_head=1),
    "16 heads": dict(n_head=16),
    "no 1/sqrt(d) scaling": dict(scale=False),
}.items():
    set_seed(0)
    m = CharTransformer(V, **kw)
    h = train_lm(m, iters=600, verbose=False)
    exp_rows[name] = {"train loss": h["train"][-1], "val loss": h["val"][-1], "seconds": h["seconds"]}
    exp_models[name] = m
display(pd.DataFrame(exp_rows).T.round(4))
print("沒有遮罩的模型生成的文字：\n" + decode(generate(exp_models["no causal mask"], torch.as_tensor([encode("ROMEO:\n")], device=device), 200, temperature=0.8)[0].tolist()))

In [ ]:
# 實驗 4：初始化時注意力權重的「尖銳程度」（entropy 越小越尖）—— 不同的 d_h
def init_att_entropy(dh: int, T: int = 64, scale: bool = True, trials: int = 20) -> float:
    g = torch.Generator().manual_seed(0)
    ents = []
    for _ in range(trials):
        q, k = torch.randn(T, dh, generator=g), torch.randn(T, dh, generator=g)
        s = q @ k.T / (math.sqrt(dh) if scale else 1.0)
        p = torch.softmax(s, dim=-1)
        ents.append(float(-(p * torch.log(p + 1e-12)).sum(-1).mean()))
    return float(np.mean(ents))


display(pd.DataFrame({dh: {"scaled": init_att_entropy(dh), "unscaled": init_att_entropy(dh, scale=False)} for dh in [8, 32, 128]}).round(3).rename_axis("d_h", axis=1))
print("均勻分布（64 個位置）的熵 = ln 64 =", round(math.log(64), 3))

In [ ]:
# 實驗 5：注意力圖（3E 訓練好的模型，第 1 層與最後一層的每個頭）
sample = "ROMEO:\nWhat light through yonder window breaks?\n\nJULIET:\nAy me!"
ids = torch.as_tensor([encode(sample)], device=device)
model.eval()
with torch.no_grad():
    model(ids)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for r, layer in enumerate([0, len(model.blocks) - 1]):
    att = model.blocks[layer].attn.last_att[0].cpu().numpy()
    for hh in range(4):
        axes[r, hh].imshow(att[hh], cmap="viridis")
        axes[r, hh].set_title(f"layer {layer + 1}, head {hh + 1}")
        axes[r, hh].set_xticks(range(len(sample)), [c if c != "\n" else "⏎" for c in sample], fontsize=5)
        axes[r, hh].set_yticks([])
plt.tight_layout()
plt.show()
att_last = model.blocks[-1].attn.last_att[0].cpu().numpy()
dist = np.abs(np.arange(len(sample))[:, None] - np.arange(len(sample))[None, :])
mean_dist = (att_last * dist[None]).sum(-1).mean(-1)
print("最後一層每個頭的「平均注意距離」（字元）：", mean_dist.round(2))

In [ ]:
# TODO 12（問答）實驗 1–5
#
# (1) 拿掉遮罩：train／val loss 變成多少？為什麼可以這麼低？生成的文字為什麼是一團亂？
#     → 這是「資料洩漏」的極端例子（第六部分第 8 章）：訓練時看得到答案，生成時看不到。
# 答：
#
# (2) 拿掉位置 embedding：val loss 差多少？模型還能學到什麼？（提示：因果遮罩本身就給了一點位置資訊 —— 每個位置「看得到幾個字元」不一樣）
# 答：
#
# (3) 頭數：1、4、16 個頭的 val loss？d_h 太小（8）有什麼問題？
# 答：
#
# (4) 不除以 √d：
#     (a) 實驗 4：d_h = 128 時，不縮放的初始注意力熵是多少？跟均勻的 ln 64 比？「太尖」對梯度有什麼影響？
#     (b) 訓練 600 步之後差多少？結果跟 (a) 的預期一致嗎？如果沒有變差，可能的原因？（提示：pre-LN 讓輸入的尺度固定、W_q／W_k 可以自己學著變小；d_h 只有 32、訓練又很短）
# 答：
#
# (5) 注意力圖：第 1 層的頭大多看「附近幾個字元」還是很遠？最後一層呢？有沒有頭在看換行或說話者的名字？
# 答：

---
## 4. 套件實作：PyTorch 內建的模組

| 手寫 | PyTorch | 對照 |
| --- | --- | --- |
| `attention_np` | `F.scaled_dot_product_attention(q, k, v, is_causal=True)` | 3C 已驗證；內建的版本會自動用 FlashAttention 等高效實作 |
| 多頭（`mha_np`、`CausalSelfAttention`） | `nn.MultiheadAttention(batch_first=True)` | 3C 已驗證 |
| `Block` | `nn.TransformerEncoderLayer(norm_first=True, activation="gelu")` + 因果遮罩 | 3E 已驗證（是的：「decoder-only」的 GPT block，用的其實是 PyTorch 的 *Encoder* layer + 因果遮罩） |

下面把 `CausalSelfAttention` 換成內建的 `F.scaled_dot_product_attention`（數學上一樣），比較速度。

In [ ]:
class FastCausalSelfAttention(CausalSelfAttention):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        h, dh = self.n_head, C // self.n_head
        q, k, v = (t.view(B, T, h, dh).transpose(1, 2) for t in self.qkv(x).split(C, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


fast = CharTransformer(V).to(device)
fast.load_state_dict(model.state_dict())
for blk_ in fast.blocks:
    blk_.attn.__class__ = FastCausalSelfAttention
x_b, _ = VAL_BATCHES[0]
with torch.no_grad():
    diff = (fast(x_b) - model(x_b)).abs().max().item()
timings = {}
for name, m in [("hand-written attention", model), ("F.scaled_dot_product_attention", fast)]:
    m.train()
    t0 = time.perf_counter()
    for _ in range(10):
        out = m(x_b)
        out.sum().backward()
    timings[name] = (time.perf_counter() - t0) / 10
print(f"✅ 兩者的 logits 差 {diff:.1e}；每步（forward + backward）：", {k: f"{v * 1000:.0f} ms" for k, v in timings.items()})

In [ ]:
# TODO 13（問答）
#
# (1) 內建的 scaled_dot_product_attention 快多少？它不需要把 (T, T) 的注意力矩陣存下來給反向傳播用（FlashAttention 的想法）—— 這對很長的 T 有什麼好處？
# (2) 為什麼 GPT（decoder-only）可以用 PyTorch 的 TransformerEncoderLayer 來做？原論文的 decoder 還多了什麼？（提示：cross-attention 看 encoder 的輸出）
# 答：

---
## 5. 調參

只用 validation（固定的 40 段）。每個設定 800 步：

| 超參數 | 試的值 |
| --- | --- |
| 學習率 | 3e-4、1e-3、3e-3 |
| 寬度 × 層數 | (128, 4)、(192, 4)、(128, 6) |

（訓練步數很少，結論只代表「這個預算下」的比較；大模型通常需要更多步才看得出優勢。）

In [ ]:
tune_rows = {}
for lr in [3e-4, 1e-3, 3e-3]:
    set_seed(0)
    m = CharTransformer(V)
    h = train_lm(m, iters=800, lr=lr, verbose=False)
    tune_rows[f"lr={lr:g}, C=128, L=4"] = {"lr": lr, "C": 128, "L": 4, "params": count_parameters(m), "val loss": h["val"][-1], "seconds": h["seconds"]}
best_lr = min((v["val loss"], v["lr"]) for v in tune_rows.values())[1]
for C, L in [(192, 4), (128, 6)]:
    set_seed(0)
    m = CharTransformer(V, C=C, n_layer=L)
    h = train_lm(m, iters=800, lr=best_lr, verbose=False)
    tune_rows[f"lr={best_lr:g}, C={C}, L={L}"] = {"lr": best_lr, "C": C, "L": L, "params": count_parameters(m), "val loss": h["val"][-1], "seconds": h["seconds"]}
tune = pd.DataFrame(tune_rows).T
display(tune)
best_row = tune.loc[tune["val loss"].astype(float).idxmin()]
BEST = {"lr": float(best_row["lr"]), "C": int(best_row["C"]), "L": int(best_row["L"])}
print("validation loss 最低：", BEST)

In [ ]:
# TODO 14（問答）
#
# (1) 三個學習率的 val loss？太小、太大各有什麼現象？
# (2) 加寬（C = 192）或加深（L = 6）：參數多了多少、時間多了多少、val loss 進步多少？在固定的步數下，大模型一定比較好嗎？
# (3) 3E 的主模型用的是 lr = 1e-3。跟這裡選出的學習率比，3E 裡「Transformer vs LSTM」的比較公平嗎？要公平比較兩種架構，還需要做什麼？（LSTM 也要調參、比同樣的時間或參數量…）
# 答：

---
## 6. 測試集評估

用選出的設定訓練 3,000 步（train 資料），在 **test**（最後 10%）上算**完整**的交叉熵：把 test 切成不重疊的 128 字元一段，每一段的每個位置都預測下一個字元。（每段開頭的字元能看到的前文比較少，所以這個數字比「每個字元都有完整 128 個前文」的評估稍差一點。）成績寫進 `results/tiny_shakespeare.csv`。

In [ ]:
from learning_map.metrics import perplexity
from learning_map.results import compare, log_result

BENCH = "tiny_shakespeare"


@torch.no_grad()
def full_ce(model: nn.Module, data: np.ndarray, T: int = 128, B: int = 64) -> float:
    model.eval()
    n_seq = (len(data) - 1) // T
    xs = data[: n_seq * T].reshape(n_seq, T)
    ys = data[1 : n_seq * T + 1].reshape(n_seq, T)
    total = 0.0
    for s in range(0, n_seq, B):
        x, y = torch.as_tensor(xs[s : s + B], device=device), torch.as_tensor(ys[s : s + B], device=device)
        total += F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1), reduction="sum").item()
    return total / (n_seq * T)


set_seed(0)
final = CharTransformer(V, C=BEST["C"], n_layer=BEST["L"])
hist_final = train_lm(final, iters=3000, lr=BEST["lr"], verbose=False)
print(f"最終模型：{count_parameters(final):,} 參數；3000 步 {hist_final['seconds']:.0f} 秒；最後的 val loss {hist_final['val'][-1]:.4f}")

test_rows = {
    "trigram (best alpha)": ngram_ce(data_tr, data_te, 3, float(val_table.index[val_table.index.str.startswith("trigram")][0].split("=")[1].rstrip(")"))),
    "LSTM 2L/256 (1500 iters)": full_ce(lstm, data_te),
    "Transformer 4L/128d (1500 iters)": full_ce(model, data_te),
    f"Transformer C={BEST['C']} L={BEST['L']} (3000 iters)": full_ce(final, data_te),
}
test_table = pd.DataFrame({k: {"test CE (nats)": v, "bits/char": v / math.log(2), **perplexity(v)} for k, v in test_rows.items()}).T
display(test_table.round(4))
for name, ce in test_rows.items():
    log_result(BENCH, name, {"ce_nats": round(ce, 4), "bpc": round(ce / math.log(2), 4), **perplexity(ce)})
compare(BENCH, "ce_nats", higher_is_better=False)
print("\n最終模型生成（temperature 0.8）：\n" + decode(generate(final, torch.as_tensor([encode("JULIET:\n")], device=device), 500, temperature=0.8)[0].tolist()))

---
## 7. 診斷：哪些字元最難猜？

1. 依「這個字元在詞裡的位置」：詞的第一個字母 vs 詞的中間 vs 空白／標點。
2. 依「前文的長度」：一段 128 字元的第 1 個位置（幾乎沒有前文）到第 128 個位置。
3. temperature：0.5、1.0、1.5 生成的文字。

In [ ]:
@torch.no_grad()
def per_char_loss(model: nn.Module, data: np.ndarray, T: int = 128, n_seq: int = 400) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    xs = data[: n_seq * T].reshape(n_seq, T)
    ys = data[1 : n_seq * T + 1].reshape(n_seq, T)
    losses = []
    for s in range(0, n_seq, 64):
        x, y = torch.as_tensor(xs[s : s + 64], device=device), torch.as_tensor(ys[s : s + 64], device=device)
        losses.append(F.cross_entropy(model(x).permute(0, 2, 1), y, reduction="none").cpu().numpy())
    return np.concatenate(losses), xs, ys


L_char, xs_, ys_ = per_char_loss(final, data_te)
prev_char = np.vectorize(ITOS.get)(xs_)
next_char = np.vectorize(ITOS.get)(ys_)
cat = np.where(~np.char.isalpha(next_char.astype(str)), "space/punct/newline", np.where(~np.char.isalpha(prev_char.astype(str)), "first letter of a word", "inside a word"))
display(pd.Series(L_char.ravel()).groupby(cat.ravel()).agg(["mean", "size"]).round(4))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, 129), L_char.mean(axis=0))
ax.set_xlabel("position in the 128-char window (amount of context)")
ax.set_ylabel("mean loss (nats)")
plt.show()
for temp in [0.5, 1.0, 1.5]:
    print(f"\n==== temperature {temp} ====\n" + decode(generate(final, torch.as_tensor([encode("KING:\n")], device=device), 200, temperature=temp)[0].tolist()))

In [ ]:
# TODO 15（問答，本章總結）
#
# (1) 測試集：trigram、LSTM、Transformer（1500 步）、最終模型的 bits/char 各是多少？
#     最終模型（調參、3000 步）有沒有贏過 LSTM（1500 步、沒調參）？為什麼可能是這樣？（字元級、小資料、短訓練：LSTM 的「循序」歸納偏誤 vs Transformer 要從資料學會「順序」）
#     test CE 比 validation 的 loss 高多少？兩個可能的原因：(a) 評估方式不同（6. 的說明）；(b) 最後 10% 是不同的劇本（分布不同）—— trigram 的 val 和 test 也差了多少？這能幫你分辨嗎？
# (2) 哪一類字元最難（詞的第一個字母、詞中間、空白標點）？為什麼「詞中間」最容易？（拼字一旦開始，後面幾乎確定）
# (3) 前文的長度：第 1 個位置的 loss 比第 128 個高多少？大約要多少個字元的前文，loss 就不太再下降？這代表模型主要用了「多遠」的資訊？
# (4) temperature：0.5、1.0、1.5 的文字有什麼差別？temperature 在數學上做了什麼？（logits / τ → τ 小越尖、τ 大越平）
# (5) 下一章 GPT：同樣的架構放大（GPT-2 small：12 層、768 維、1.24 億參數）、改用 BPE 子詞、在大量網路文字上預訓練。你預期載入預訓練權重後，在 Tiny Shakespeare 上微調，bits/char 會好多少？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Tiny Shakespeare（字元級；train 80%、val 10%、test 10%）

| 模型 | 參數 | 訓練步數 | val CE | test CE | test bits/char |
| --- | --- | --- | --- | --- | --- |
| 均勻／unigram／bigram／trigram | | — | | | |
| LSTM 2L/256 | | 1500 | | | |
| Transformer 4L/128d | | 1500 | | | |
| Transformer（調參後） | | 3000 | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ Transformer 的反向傳播（手寫 NumPy 版的 attention backward）
- ☐ 位置編碼的變體：相對位置、RoPE（旋轉位置編碼）、ALiBi
- ☐ 注意力的計算與記憶體：FlashAttention、KV cache（生成時不用重算前面的 K、V）
- ☐ pre-LN vs post-LN 的訓練穩定性；權重共享（tok_emb 與 head）

**延伸**
- ☐ 原論文的 encoder-decoder 與 cross-attention（第四部分 Seq2Seq；本部分 T5、BART）
- ☐ 子詞切分（BPE）：下一章 GPT
- ☐ Scaling laws（Kaplan et al. 2020）：參數、資料、計算量與 loss 的關係